In [ ]:
%cd ../src

In [ ]:
import polars as pl
from compute import agg_scan_results
from config.compute import ScannerConfig
from config.file_paths import FilePathConfig
from nse import NSE

In [ ]:
nse_api = NSE(".")

In [ ]:
END_DATE = "2026-11-27"
file_paths = FilePathConfig(END_DATE)

In [ ]:
long_res, short_res = agg_scan_results(scan_conf=ScannerConfig, file_paths=file_paths)

In [ ]:
long_res

In [ ]:
total_watchlist = (
    long_res.filter(pl.col("rs_flag") & pl.col("scan_flag"))
    .select(pl.col("symbol").n_unique())
    .item()
)


(
    long_res.lazy()
    .filter(pl.col("rs_flag") & pl.col("scan_flag"))
    .group_by("sector")
    .agg(
        pl.col("symbol").len().alias("stock_count"),
        pl.col("rs").mean().round().alias("rs"),
        pl.col("rs_prev_week").mean().round().alias("rs_prev_week"),
    )
    .with_columns(
        ((pl.col("rs") - pl.col("rs_prev_week")) * 100 / pl.col("rs_prev_week"))
        .clip(upper_bound=100)
        .round(2)
        .alias("rs_pct_change"),
        (pl.col("stock_count") * 100 / total_watchlist)
        .round(2)
        .alias("watchlist_contribution_pct"),
    )
    .with_columns(
        (0.40 * pl.col("rs") + 0.60 * pl.col("watchlist_contribution_pct"))
        .round(2)
        .alias("score")
    )
    .sort("score", descending=True)
    .collect()
).to_pandas()

In [ ]:
short_res.filter(pl.col("fno_flag") & pl.col("rs_flag"))